# Siete casos, ocho objetivos: matching bipartito KISS

**Pregunta.** ¿Cómo asignar, uno a uno, escenarios clínico-educativos simulados compatibles a microobjetivos de aprendizaje sobre severidad para maximizar los objetivos cubiertos, sin reutilizar casos?

Este notebook es educativo. No diagnostica, prescribe ni trabaja con pacientes.

## 1. Traducción a grafo

- `U`: siete escenarios simbólicos, uno por criterio explícito de severidad.
- `V`: siete microobjetivos de identificación y un microobjetivo de ruta/BPC.
- `E`: cada caso se conecta con su criterio y con la BPC común.
- `M`: pares seleccionados sin repetir caso ni objetivo.

Cubrir significa **asignar un representante compatible**, no demostrar aprendizaje o validez clínica.

In [ ]:
# Ejecutar solo en Google Colab. En local, instale el repositorio con pip install -e '.[dev]'.
import os
import subprocess
import sys

if 'COLAB_RELEASE_TAG' in os.environ and not os.path.exists('/content/grafos-LLM'):
    subprocess.run([
        'git', 'clone', '--branch', 'research-var-cases', '--single-branch',
        'https://github.com/llabans/grafos-LLM.git', '/content/grafos-LLM'
    ], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/grafos-LLM[dev]'], check=True)

In [ ]:
from bipartite_minsa.kiss_case import (
    build_kiss_severity_graph, case_labels, objective_labels,
)
from bipartite_minsa.matching.analysis import hall_deficiency_witnesses
from bipartite_minsa.matching.augmenting_path import AugmentingPathMatcher
from bipartite_minsa.matching.hopcroft_karp import HopcroftKarpMatcher
from bipartite_minsa.matching.oracles import networkx_cardinality

graph = build_kiss_severity_graph()
print(f'|U|={len(graph.left)} |V|={len(graph.right)} |E|={graph.m}')

In [ ]:
case_names = case_labels()
objective_names = objective_labels()

for case_id in graph.left:
    neighbors = [objective_names[v] for v in graph.adjacency[case_id]]
    print(f'{case_names[case_id]} -> {neighbors}')

## 2. Resolver y triangular

Usamos dos implementaciones propias y NetworkX únicamente como oráculo externo de cardinalidad.

In [ ]:
dfs = AugmentingPathMatcher().solve(graph)
hk = HopcroftKarpMatcher().solve(graph)
oracle = networkx_cardinality(graph)

print(f'DFS={dfs.cardinality} HK={hk.cardinality} NetworkX={oracle}')
assert dfs.cardinality == hk.cardinality == oracle == 7

In [ ]:
matched_objectives = {right for _, right in hk.pairs}
uncovered = sorted(set(graph.right) - matched_objectives)
print('Objetivo no representado en este óptimo:', objective_names[uncovered[0]])

witness = hall_deficiency_witnesses(graph, hk)[0]
print(f'Hall: |S|={len(witness.target_ids)} > |N(S)|={len(witness.neighboring_scenario_ids)}')
assert (len(witness.target_ids), len(witness.neighboring_scenario_ids)) == (8, 7)

## 3. Interpretación

La solución factible tiene siete pares y ningún matching puede usar más de siete casos. Hall explica la misma brecha: ocho objetivos compiten por siete vecinos.

El objetivo concreto que queda fuera depende del desempate y **no representa una prioridad clínica**. Si un caso pudiera acreditar varios objetivos, el problema cambiaría a cobertura o `b`-matching.

## 4. Estado científico

- Algoritmo y certificado estructural: validados.
- Vínculo con literatura de GPC computables, grafos y simulación: documentado.
- Coherencia clínica de viñetas completas y efecto educativo: requieren panel y estudio prospectivo.

**LOOP:** definir → construir → resolver → verificar → validar → interpretar → repetir.